# Successive Over-Relaxation (SOR) Method
The successive over-relaxation method introduces a relaxation factor $\omega$ into the [Gauss–Seidel update](CHEME-5800-L6c-Algorithm-GaussSeidel-Fall-2026.ipynb). Each component is updated in sequence, and $\omega$ controls how far it moves toward, or beyond, the value obtained by solving its row equation. A suitable choice can accelerate convergence; the benefit depends on the matrix and the ordering of the unknowns.

Consider $\mathbf{A}\mathbf{x}=\mathbf{b}$, where $\mathbf{A}\in\mathbb{R}^{n\times n}$ is nonsingular with nonzero diagonal entries, $\mathbf{x}$ contains the unknowns, and $\mathbf{b}$ is the right-hand side. Let $\mathbf{D}$, $\mathbf{L}$, and $\mathbf{U}$ contain the diagonal, strictly lower triangular, and strictly upper triangular entries of $\mathbf{A}$. For $0<\omega<2$, the splitting $\mathbf{A}=\mathbf{M}_{\omega}-\mathbf{N}_{\omega}$ is given by:

$$
\begin{aligned}
\mathbf{M}_{\omega}&=\frac{1}{\omega}(\mathbf{D}+\omega\mathbf{L}),\\
\mathbf{N}_{\omega}&=\left(\frac{1}{\omega}-1\right)\mathbf{D}-\mathbf{U}.
\end{aligned}
$$

At iteration $k$, we compute the correction $\mathbf{d}^{(k)}$ from the residual as follows:

$$
\boxed{
(\mathbf{D}+\omega\mathbf{L})\mathbf{d}^{(k)}
=\omega\,\underbrace{(\mathbf{b}-\mathbf{A}\mathbf{x}^{(k)})}_{\text{residual }\mathbf{r}^{(k)}}.
}
$$

This lower triangular system is solved by forward substitution. Applying the correction gives the equivalent component update:

$$
x_i^{(k+1)}=(1-\omega)x_i^{(k)}
+\frac{\omega}{a_{ii}}\left(b_i-\sum_{j<i}a_{ij}x_j^{(k+1)}-\sum_{j>i}a_{ij}x_j^{(k)}\right),
\qquad i=1,\ldots,n.
$$

The first sum uses values already updated during the sweep; the second uses values from the previous iteration. With $0<\omega<1$, we take a partial step (__under-relaxation__); $\omega=1$ recovers Gauss–Seidel; and $1<\omega<2$ takes the component beyond its row-equation value (__over-relaxation__). The interval alone does not guarantee convergence for every matrix.

Let's sketch out the SOR method.

__Initialize__: Given the system matrix $\mathbf{A}\in\mathbb{R}^{n\times n}$ and right-hand side $\mathbf{b}\in\mathbb{R}^{n}$, choose an initial guess $\mathbf{x}^{(0)}\in\mathbb{R}^{n}$, an absolute residual tolerance $\epsilon>0$, and a nonnegative integer correction limit $\texttt{maxiter}$. Set $\texttt{converged}\gets\texttt{false}$ and the correction counter $k\gets0$. Require nonzero diagonal entries of $\mathbf{A}$ and choose a relaxation factor $\omega\in(0,2)$.

While not $\texttt{converged}$ __do__:

1. Calculate the residual vector $\mathbf{r}^{(k)}\gets\mathbf{b}-\mathbf{A}\mathbf{x}^{(k)}$.
2. Check for convergence:
   - If $\|\mathbf{r}^{(k)}\|_2<\epsilon$, __then__: set $\texttt{converged}\gets\texttt{true}$ and return $\mathbf{x}^{(k)}$ with this status.
   - Otherwise, if $k\ge\texttt{maxiter}$, __then__: print a __warning__ that the residual tolerance was not met and return $\mathbf{x}^{(k)}$ with $\texttt{converged}=\texttt{false}$.
3. Calculate the update direction: solve $(\mathbf{D}+\omega\mathbf{L})\mathbf{d}^{(k)}=\omega\mathbf{r}^{(k)}$ by forward substitution.
4. Update the solution vector: $\mathbf{x}^{(k+1)}\gets\mathbf{x}^{(k)}+\mathbf{d}^{(k)}$.
5. Increment the correction counter: $k\gets k+1$.

Either return stops the algorithm immediately. Checking the residual first recognizes convergence at the initial guess or after the last allowed correction.

### Convergence and choosing $\omega$
The stationary update has the following iteration matrix:

$$
\mathbf{G}_{\omega}
=(\mathbf{D}+\omega\mathbf{L})^{-1}\bigl((1-\omega)\mathbf{D}-\omega\mathbf{U}\bigr).
$$

The iteration converges to the solution from every initial guess if and only if $\rho(\mathbf{G}_{\omega})<1$, where $\rho$ is the largest eigenvalue magnitude. For a [symmetric positive definite matrix](https://netlib.org/linalg/html_templates/node16.html), convergence is guaranteed for every $0<\omega<2$. This guarantee does not identify the fastest choice.

A useful special case is a **symmetric positive definite tridiagonal matrix**, whose entries satisfy $a_{ij}=0$ when $|i-j|>1$. In the natural row order, the relaxation factor that minimizes the spectral radius is given by the [classical optimal-parameter formula](https://faculty.washington.edu/trogdon/105A/html/Lecture20.html):

$$
\omega_{\mathrm{opt}}
=\frac{2}{1+\sqrt{1-\rho(\mathbf{G}_{J})^2}},
\qquad \mathbf{G}_{J}=-\mathbf{D}^{-1}(\mathbf{L}+\mathbf{U}),
$$

where $\mathbf{G}_{J}$ is the Jacobi iteration matrix. Here “optimal” refers to the asymptotic convergence factor; it does not guarantee the fewest iterations for every right-hand side, initial guess, and stopping tolerance. Positive definiteness alone is not enough to apply this formula.

For a general matrix, compare candidate relaxation factors using the same initial guess and residual tolerance, and verify convergence before comparing iteration counts or timings. The [companion example](CHEME-5800-L6c-Example-FunWithIterativeSolvers-Fall-2026.ipynb) provides a system for this comparison.

___

## Summary
We developed a relaxed version of the sequential Gauss–Seidel update.

> __Key Takeaways:__
>
> - **Relaxed sequential updates:** We used a relaxation factor to control each component's step while retaining the newest available values during the sweep.
> - **Convergence depends on the matrix:** We distinguished the positive-definite convergence guarantee from the additional structure needed for the optimal-parameter formula.
> - **Parameter comparisons need a common stopping rule:** We separated successful convergence from reaching a correction limit and established a common residual tolerance for comparing relaxation choices.

The companion example applies these ideas to a numerical system and measures solver performance.

___
